In [ ]:
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Iterable
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

PROJECT = Path("..").resolve()
DATA = PROJECT / "data"
ELAN_DIR = DATA / "annotations" / "ELAN"
PROCESSED = DATA / "processed"
DIAR_DIR = PROCESSED / "diarization"
OUT = DATA / "out"
REPORTS = OUT / "reports"
DATASET = OUT / "dataset"
LOGS = OUT / "logs"

for p in [REPORTS, DATASET, LOGS]:
    p.mkdir(parents=True, exist_ok=True)

ACTIVE_CONSTRUCTS = ["CH", "IS", "MF", "OI", "SJ", "SK"]

TIER_TO_LABEL = {
    "self_kindness": "SK",
    "self_judgement": "SJ",
    "common_humanity": "CH",
    "isolation": "IS",
    "mindfulness": "MF",
    "over_identified": "OI",
}

def canonical_label_from_tier(tier_id: str) -> str | None:
    t = tier_id.lower()
    for key, val in TIER_TO_LABEL.items():
        if key in t:
            return val
    return None


MAX_GAP_SEC = 0.8   
MIN_SEG_SEC  = 1.0  
TAU_KEEP = 0.20     
CLIP_EPS = 0.02     

MIN_ABS_POS_BY_CLASS = {
    "CH": 0.1,
    "IS": 0.2,
    "MF": 1.0,
    "OI": 0.2,
    "SJ": 1.0,
    "SK": 1.0,
}

TAU_KEEP_BY_CLASS = {
    "CH": 0.05,   
    "IS": 0.10,
    "SJ": 0.15,
    "OI": 0.10,
}

COVERAGE_MIN_BY_CLASS = {
    "CH": 0.00,
    "IS": 0.00,
    "MF": 0.01,
    "OI": 0.01,
    "SJ": 0.01,
    "SK": 0.01,
}

def constructs_list() -> list[str]:
    return list(ACTIVE_CONSTRUCTS)


In [ ]:
@dataclass
class Interval:
    start: float
    end: float
    label: str | None = None
    tier: str | None = None
    meta: dict | None = None

def overlap(a: Interval, b: Interval) -> float:
    return max(0.0, min(a.end, b.end) - max(a.start, b.start))

def merge_with_gaps(intervals: List[Interval], max_gap: float) -> List[Interval]:
    if not intervals:
        return []
    ivs = sorted(intervals, key=lambda x: (x.start, x.end))
    out = [Interval(ivs[0].start, ivs[0].end, meta={"members":[ivs[0]]})]
    for iv in ivs[1:]:
        last = out[-1]
        gap = iv.start - last.end
        if gap <= max_gap:
            last.end = max(last.end, iv.end)
            last.meta["members"].append(iv)
        else:
            out.append(Interval(iv.start, iv.end, meta={"members":[iv]}))
    return out

def merge_union(intervals: List[Interval], max_gap: float = 0.0) -> List[Interval]:
    if not intervals:
        return []
    ivs = sorted(intervals, key=lambda x: (x.start, x.end))
    out = [Interval(ivs[0].start, ivs[0].end)]
    for iv in ivs[1:]:
        last = out[-1]
        if iv.start <= last.end + max_gap:
            last.end = max(last.end, iv.end)
        else:
            out.append(Interval(iv.start, iv.end))
    return out

def total_length(intervals: List[Interval]) -> float:
    return sum(max(0.0, iv.end - iv.start) for iv in intervals)

def intersection_length(a: List[Interval], b: List[Interval]) -> float:
    i = j = 0; tot = 0.0
    while i < len(a) and j < len(b):
        s = max(a[i].start, b[j].start)
        e = min(a[i].end, b[j].end)
        if e > s:
            tot += (e - s)
        if a[i].end < b[j].end: i += 1
        else: j += 1
    return tot


In [ ]:
def parse_eaf_intervals(eaf_path: Path) -> List[Interval]:
    tree = ET.parse(eaf_path)
    root = tree.getroot()
    ns = {"elan": root.tag.split("}")[0].strip("{")} if "}" in root.tag else {}

    time_order = root.find("elan:TIME_ORDER", ns) if ns else root.find("TIME_ORDER")
    ts_map = {}
    for ts in (time_order.findall("elan:TIME_SLOT", ns) if ns else time_order.findall("TIME_SLOT")):
        ts_id = ts.attrib["TIME_SLOT_ID"]
        tval = float(ts.attrib["TIME_VALUE"]) / 1000.0  # ms -> sec
        ts_map[ts_id] = tval

    intervals = []
    for tier in (root.findall("elan:TIER", ns) if ns else root.findall("TIER")):
        tier_id = tier.attrib.get("TIER_ID", "")
        anns = tier.findall("elan:ALIGNABLE_ANNOTATION", ns) if ns else tier.findall("ALIGNABLE_ANNOTATION")
        for ann in anns:
            t1 = ts_map.get(ann.attrib["TIME_SLOT_REF1"])
            t2 = ts_map.get(ann.attrib["TIME_SLOT_REF2"])
            if t1 is None or t2 is None or t2 <= t1:
                continue
            aval_el = ann.find("elan:ANNOTATION_VALUE", ns) if ns else ann.find("ANNOTATION_VALUE")
            label_text = (aval_el.text or "").strip() if aval_el is not None else ""
            canon = canonical_label_from_tier(tier_id)
            intervals.append(Interval(start=t1, end=t2, label=canon or label_text, tier=tier_id))
    return intervals

def load_elan_for_video(video_id: str, elan_dir: Path = ELAN_DIR) -> List[Interval]:
    exact = elan_dir / f"{video_id}.eaf"
    if exact.exists():
        return parse_eaf_intervals(exact)

    pref = sorted(elan_dir.glob(f"{video_id}*.eaf"))
    if pref:
        return parse_eaf_intervals(pref[0])

    vi_lower = video_id.lower()
    candidates = sorted(elan_dir.glob("*.eaf"))
    for p in candidates:
        stem = p.stem.lower()
        if stem == vi_lower or stem.startswith(vi_lower + "_") or stem.startswith(vi_lower + "-"):
            return parse_eaf_intervals(p)

    for p in candidates:
        try:
            tree = ET.parse(p)
            root = tree.getroot()
            ns = {"elan": root.tag.split("}")[0].strip("{")} if "}" in root.tag else {}
            header = root.find("elan:HEADER", ns) if ns else root.find("HEADER")
            if header is None:
                continue
            mds = header.findall("elan:MEDIA_DESCRIPTOR", ns) if ns else header.findall("MEDIA_DESCRIPTOR")
            for md in mds:
                for attr in ("MEDIA_URL", "RELATIVE_MEDIA_URL"):
                    val = md.attrib.get(attr)
                    if val and vi_lower in val.lower():
                        return parse_eaf_intervals(p)
        except Exception:
            pass

    raise FileNotFoundError(f"No ELAN")


In [ ]:
def load_elan_for_video(video_id: str, elan_dir: Path = ELAN_DIR) -> List[Interval]:
    exact = elan_dir / f"{video_id}.eaf"
    if exact.exists():
        return parse_eaf_intervals(exact)

    pref = sorted(elan_dir.glob(f"{video_id}*.eaf"))
    if pref:
        return parse_eaf_intervals(pref[0])

    vi_lower = video_id.lower()
    candidates = sorted(elan_dir.glob("*.eaf"))
    for p in candidates:
        stem = p.stem.lower()
        if stem == vi_lower or stem.startswith(vi_lower + "_") or stem.startswith(vi_lower + "-"):
            return parse_eaf_intervals(p)

    for p in candidates:
        try:
            tree = ET.parse(p)
            root = tree.getroot()
            ns = {"elan": root.tag.split("}")[0].strip("{")} if "}" in root.tag else {}
            header = root.find("elan:HEADER", ns) if ns else root.find("HEADER")
            if header is None:
                continue
            mds = header.findall("elan:MEDIA_DESCRIPTOR", ns) if ns else header.findall("MEDIA_DESCRIPTOR")
            for md in mds:
                for attr in ("MEDIA_URL", "RELATIVE_MEDIA_URL"):
                    val = md.attrib.get(attr)
                    if val and vi_lower in val.lower():
                        return parse_eaf_intervals(p)
        except Exception:
            
            pass

    raise FileNotFoundError(f"No ELAN")
6
def load_rttm_intervals(rttm_path: Path) -> List[Interval]:
    ivs = []
    with open(rttm_path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("
                continue
            parts = line.split()
            if len(parts) < 9 or parts[0] != "SPEAKER":
                continue
            start = float(parts[3]); dur = float(parts[4]); spk = parts[7]
            ivs.append(Interval(start=start, end=start+dur, meta={"speaker": spk}))
    return ivs

def load_diar_csv(csv_path: Path) -> List[Interval]:
    df = pd.read_csv(csv_path)
    
    def find_col(candidates):
        for c in df.columns:
            if c.lower() in candidates:
                return c
        raise ValueError(f"Missing any of columns {candidates} in {csv_path.name}")
    c_start = find_col({"start","t_start","begin"})
    c_end   = find_col({"end","t_end","stop"})
    c_spk   = None
    for col in df.columns:
        if col.lower() in {"speaker","spk","label","who"}:
            c_spk = col
            break

    ivs = []
    for _, row in df.iterrows():
        s = float(row[c_start]); e = float(row[c_end])
        if e > s:
            meta = {}
            if c_spk is not None:
                meta["speaker"] = str(row[c_spk])
            ivs.append(Interval(start=s, end=e, meta=meta))
    return ivs

def load_diarization_for_video(video_id: str) -> List[Interval]:
    candidates_csv = sorted(DIAR_DIR.glob(f"{video_id}*.csv"))
    candidates_rttm = sorted(DIAR_DIR.glob(f"{video_id}*.rttm"))
    if candidates_csv:
        return load_diar_csv(candidates_csv[0])
    if candidates_rttm:
        return load_rttm_intervals(candidates_rttm[0])
    raise FileNotFoundError(f"No diarization found for {video_id} in {DIAR_DIR}")



In [ ]:
def load_rttm_intervals(rttm_path: Path) -> List[Interval]:
    ivs = []
    with open(rttm_path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 9 or parts[0] != "SPEAKER":
                continue
            start = float(parts[3]); dur = float(parts[4]); spk = parts[7]
            ivs.append(Interval(start=start, end=start+dur, meta={"speaker": spk}))
    return ivs

def load_diar_csv(csv_path: Path) -> List[Interval]:
    df = pd.read_csv(csv_path)
    def find_col(cands):
        for c in df.columns:
            if c.lower() in cands:
                return c
        raise ValueError(f"Missing any of columns {cands} in {csv_path.name}")
    c_start = find_col({"start","t_start","begin"})
    c_end   = find_col({"end","t_end","stop"})
    c_spk   = None
    for col in df.columns:
        if col.lower() in {"speaker","spk","label","who"}:
            c_spk = col; break

    ivs = []
    for _, row in df.iterrows():
        s = float(row[c_start]); e = float(row[c_end])
        if e > s:
            meta = {}
            if c_spk is not None:
                meta["speaker"] = str(row[c_spk])
            ivs.append(Interval(start=s, end=e, meta=meta))
    return ivs

def load_diarization_for_video(video_id: str) -> List[Interval]:
    candidates_csv = sorted(DIAR_DIR.glob(f"{video_id}*.csv"))
    candidates_rttm = sorted(DIAR_DIR.glob(f"{video_id}*.rttm"))
    if candidates_csv:
        return load_diar_csv(candidates_csv[0])
    if candidates_rttm:
        return load_rttm_intervals(candidates_rttm[0])
    raise FileNotFoundError(f"No diarization found for {video_id} in {DIAR_DIR}")


In [ ]:
def union_by_construct(label_intervals: List[Interval]) -> Dict[str, List[Interval]]:
    by_construct: Dict[str, List[Interval]] = {}
    for iv in label_intervals:
        canon = canonical_label_from_tier(iv.tier or "") or (iv.label if iv.label in set(TIER_TO_LABEL.values()) else None)
        if not canon:
            continue
        by_construct.setdefault(canon, []).append(iv)
    return {c: merge_union(ivs, max_gap=0.0) for c, ivs in by_construct.items()}

def compute_tmax(video_id: str) -> float:
    ends = []
    try:
        for iv in load_diarization_for_video(video_id):
            ends.append(iv.end)
    except Exception:
        pass
    try:
        for iv in load_elan_for_video(video_id):
            ends.append(iv.end)
    except Exception:
        pass
    if not ends:
        raise ValueError(f"No timing sources found for {video_id}.")
    return max(ends)

def all_speech_union(diar_ivs: List[Interval]) -> List[Interval]:
    return merge_union(diar_ivs, max_gap=0.0)

def segment_speech_blocks_all(diar_ivs: List[Interval]) -> List[Interval]:
    merged = merge_with_gaps(diar_ivs, MAX_GAP_SEC)
    return [Interval(m.start, m.end) for m in merged if (m.end - m.start) >= MIN_SEG_SEC]

def complement_silence_segments(speech: List[Interval], tmax: float) -> List[Interval]:
    sil = []
    cur = 0.0
    for iv in sorted(speech, key=lambda x: (x.start, x.end)):
        if iv.start > cur:
            if (iv.start - cur) >= MIN_SEG_SEC:
                sil.append(Interval(cur, iv.start))
        cur = max(cur, iv.end)
    if tmax > cur and (tmax - cur) >= MIN_SEG_SEC:
        sil.append(Interval(cur, tmax))
    return sil


In [ ]:
def compute_soft_targets_for_video(video_id: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    diar_ivs = load_diarization_for_video(video_id)
    tmax = compute_tmax(video_id)
    if tmax <= 0:
        raise ValueError(f"[{video_id}] Invalid timeline length.")

    speech_segments = segment_speech_blocks_all(diar_ivs)
    speech_segments = [s for s in speech_segments if s.end <= tmax]
    speech_union_strict = all_speech_union(diar_ivs)
    silence_segments = complement_silence_segments(speech_union_strict, tmax)

    segments: List[Interval] = []
    for s in speech_segments:
        seg = Interval(s.start, s.end, meta={"segment_type":"speech"}); segments.append(seg)
    for s in silence_segments:
        seg = Interval(s.start, s.end, meta={"segment_type":"silence"}); segments.append(seg)
    segments.sort(key=lambda iv: (iv.start, iv.end))
    if not segments:
        raise ValueError(f"[{video_id}] No segments after gap/min-length rules.")

    elan_ivs = load_elan_for_video(video_id)
    by_construct = union_by_construct(elan_ivs)
    constructs = constructs_list()
    for c in constructs:
        by_construct.setdefault(c, [])
    
    timeline_sec = tmax
    cov_rows = []
    for c in constructs:
        ann_union = merge_union(by_construct[c])
        clipped = [Interval(max(0.0, iv.start), min(tmax, iv.end)) for iv in ann_union if min(tmax, iv.end) > max(0.0, iv.start)]
        ann_sec = total_length(clipped)
        cov = ann_sec / timeline_sec if timeline_sec > 0 else 0.0
        cov_rows.append(dict(video_id=video_id, construct=c, annotated_sec=round(ann_sec,3),
                             timeline_sec=round(timeline_sec,3), coverage=round(cov,4)))
    coverage_df = pd.DataFrame(cov_rows)
    
    rows = []
    for i, seg in enumerate(segments):
        dur = seg.end - seg.start
        seg_type = (seg.meta or {}).get("segment_type","speech")
        row = dict(video_id=video_id, seg_id=f"{i:05d}", t_start=round(seg.start,3), t_end=round(seg.end,3),
                   duration_sec=round(dur,3), segment_type=seg_type)
        for c in constructs:
            ov = 0.0
            for iv in by_construct[c]:
                ov += overlap(seg, iv)
            p = ov / dur if dur > 0 else 0.0
            min_abs = MIN_ABS_POS_BY_CLASS.get(c, 1.0)
            tau_keep_c = TAU_KEEP_BY_CLASS.get(c, TAU_KEEP)
            if p >= tau_keep_c and ov >= min_abs:
                y = max(CLIP_EPS, min(1.0 - CLIP_EPS, p))
            else:
                y = 0.0
            row[f"y_{c}"] = round(y, 4)
        rows.append(row)
    targets_df = pd.DataFrame(rows)

    
    cov_map = {}
    for _, r in coverage_df.iterrows():
        c = r["construct"]
        thr = COVERAGE_MIN_BY_CLASS.get(c, 0.01)
        cov_map[c] = 1 if (r["annotated_sec"] > 0 and r["coverage"] >= thr) else 0
    for c in constructs:
        targets_df[f"mask_{c}"] = cov_map.get(c, 0)

    
    per_video_segments = DATASET / f"{video_id}_segments_master.csv"
    per_video_targets  = DATASET / f"{video_id}_targets_multi.csv"
    targets_df[["video_id","seg_id","t_start","t_end","duration_sec","segment_type"]].to_csv(per_video_segments, index=False)
    targets_df.to_csv(per_video_targets, index=False)
    print(f"[OK] {video_id}: segments → {per_video_segments.name} ({len(targets_df)} rows); targets → {per_video_targets.name}")

    return targets_df[["video_id","seg_id","t_start","t_end","duration_sec","segment_type"]], targets_df, coverage_df


In [ ]:
def list_video_ids_from_diar(diar_dir: Path = DIAR_DIR) -> List[str]:
    ids = set()
    for p in diar_dir.glob("*.csv"):
        ids.add(p.stem.split("_")[0])
    for p in diar_dir.glob("*.rttm"):
        ids.add(p.stem.split("_")[0])
    return sorted(ids)

def build_dataset_multi_label(video_ids: List[str] | None = None):
    if video_ids is None:
        video_ids = list_video_ids_from_diar()
    all_segs = []
    all_targets = []
    all_cov = []
    for vid in video_ids:
        try:
            seg_df, tgt_df, cov_df = compute_soft_targets_for_video(vid)
            all_segs.append(seg_df)
            all_targets.append(tgt_df)
            all_cov.append(cov_df)
        except Exception as e:
            print(f"[ERROR] {vid}: {e}")

    if not all_segs or not all_targets:
        print("[ABORT] No outputs generated.")
        return

    segs_all = pd.concat(all_segs, ignore_index=True)
    tgts_all = pd.concat(all_targets, ignore_index=True)
    cov_all = pd.concat(all_cov, ignore_index=True) if all_cov else pd.DataFrame()

    segs_all.to_csv(DATASET / "segments_master.csv", index=False)
    tgts_all.to_csv(DATASET / "targets_multi.csv", index=False)
    cov_all.to_csv(REPORTS / "coverage_by_video_construct.csv", index=False)

    if not cov_all.empty:
        print(f"coverage_by_video")


In [ ]:
TEST_VIDEO_ID = "MP00004"

try:
    seg_df, tgt_df, cov_df = compute_soft_targets_for_video(TEST_VIDEO_ID)
    print("\n[Preview] First 8 segments with soft labels:")
    cols_show = ["video_id","seg_id","t_start","t_end","duration_sec","segment_type"] + [f"y_{c}" for c in constructs_list()] + [f"mask_{c}" for c in constructs_list()]
    try:
        display(tgt_df[cols_show].head(8))
    except Exception:
        print(tgt_df[cols_show].head(8))
    print("\n[Coverage]")
    try:
        display(cov_df)
    except Exception:
        print(cov_df.head())
except Exception as e:
    print(f"[TEST ERROR] {e}")


In [ ]:
build_dataset_multi_label()